<a id='introduc'></a>

## Introduction



<a id='top'></a>

# Test Reference files in MIRI FWSS Data. 





<a id='test_data'></a>

## Test  data




<a id='pipe_doc'></a>



<a id='ref_files'></a>

### Reference files

The JWST pipeline makes use of a large set of FITS and ASDF [reference files](https://jwst-pipeline.readthedocs.io/en/latest/jwst/references_general/references_general.html#reference-file-types) employed to perform different corrections and calibrations.

When a pipeline module is executed, it will first check whether the required reference files for that module are present in our local reference files directory. If that is not the case, they will be automatically downloaded from the [Calibration Reference Data System (CRDS)](https://jwst-crds.stsci.edu/) as long as we have correctly set a couple of environment variables: `CRDS_PATH` defines the folder where the reference files will be stored (you can change it to any folder you prefer), and `CRDS_SERVER_URL` contains the CRDS URL direction from where the reference files will be downloaded.

>`$ export CRDS_PATH=$HOME/crds_cache`<br>
>`$ export CRDS_SERVER_URL=https://jwst-crds.stsci.edu`

<a id='import'></a>

## Imports

In [1]:
import os
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from astropy.visualization import (MinMaxInterval, SqrtStretch,
                                   ImageNormalize)

from scipy.constants import c
from scipy.stats import sigmaclip

import astropy.units as u
from astropy.io import fits
from astropy.io.fits import table_to_hdu
from astropy.table import Table
from astropy.visualization import ImageNormalize

from stdatamodels.jwst import datamodels
from stdatamodels.jwst.datamodels import dqflags
from jwst.datamodels import ModelContainer

from jwst.pipeline import Spec2Pipeline
from astropy.stats import sigma_clipped_stats as sigclip
import jwst

import copy
from glob import glob
from matplotlib import rc
rc('text', usetex=True)

INPUT DATA:
jw09505001001_02101_00001_mirimage_cal.fits - Image
jw09505001002_02101_00001_mirimage_cal.fits - Image
jw09505001002_02101_00002_mirimage_cal.fits - Image 
jw09505001003_02101_00001_mirimage_cal.fits - Image


jw09505001001_02102_00001_mirimage_cal.fits - spec
jw09505001001_02102_00002_mirimage_cal.fits - spec
jw09505001001_02102_00003_mirimage_cal.fits - spec
jw09505001001_02102_00004_mirimage_cal.fits- spec

In [5]:
input_obs_dir = '/Users/morrison/MIRI_WFSS/MIRI9505/'
stage1_dir = input_obs_dir
stage2_dir = input_obs_dir


In [9]:
# settting up running calspec2 using an association that contains
# science file
# direct image file: rate file from image2
# source cat and  segmap from calimage3
import logging
log = logging.getLogger()
log.setLevel('INFO')
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
handler = logging.StreamHandler()
handler.setFormatter(formatter)
log.addHandler(handler)
# # Run spec2 pipeline on WFSS
# until WFSS is a mode need to override reference files 


def run_spec2(asn):
    from jwst.pipeline import Spec2Pipeline
    from glob import glob
    from jwst.assign_wcs import AssignWcsStep
    from jwst.extract_2d import Extract2dStep

    from jwst.photom import PhotomStep
    from jwst.extract_1d import Extract1dStep
    
    region_file = 'N/A'
    photom_file = 'jwst_miri_photom_WFSS_20260220_v2.fits'
    foffset_file = 'jwst_miri_filteroffset_0008.asdf'
    flat_file = 'jwst_miri_flat_WFSS.fits' # needs testing 
    extract1d_file = 'N/A' # not needed for WFSS
    apcorr_file = 'N/A' # not needed for WFSS
    wavelengthrange_file = 'miri_wfss_wavelengthrange.asdf'
    specwcs = 'MIRI_WFSS_specwcs_20260212.asdf'
    bkg_file = 'MIRI_WFSS_bkg_February2026.fits'
    
    spec2 = Spec2Pipeline()
    spec2.assign_wcs.override_filteroffset= foffset_file
    spec2.assign_wcs.override_specwcs = specwcs
    spec2.assign_wcs.skip= False
    spec2.extract_2d.save_results = True
    spec2.extract_1d.save_results = True
    spec2.extract_2d.wfss_extract_half_height=25
    spec2.save_results = True
    spec2.photom.override_photom = photom_file
    spec2.photom.save_results = True
    spec2.extract_1d.override_extract1d = extract1d_file
    spec2.extract_1d.override_apcorr = apcorr_file
    spec2.bkg_subtract.skip=  False
    spec2.bkg_subtract.override_bkg = bkg_file
    spec2.bkg_subtract.save_results = True
    spec2.photom.save_results = True
    spec2.flat_field.save_results = True
    spec2.flat_field.skip = False
    spec2.flat_field.override_flat = flat_file
    result = spec2.run(asn) 
    


In [10]:
# Create the spec2 association for the MIR_FWSS data = create by hand
# change exp_type in header of MIR_WFSS to MIR_WFSS of the FITs HEADER of WFSS data 

print('Importing JWST pipeline version {}'.format(jwst.__version__))


Importing JWST pipeline version 1.21.0.dev187+ged9f2af08


In [11]:
asn_spec2 = sorted(glob('*spec2*_asn.json'))
#asn1 = asn_spec2[0]
#print(asn1)

print(asn_spec2)



['jw09505-o001_spec2_00001_asn.json', 'jw09505-o001_spec2_00002_asn.json', 'jw09505-o001_spec2_00003_asn.json', 'jw09505-o001_spec2_00004_asn.json']


In [12]:
for asn in asn_spec2:
    run_spec2(asn)

2026-03-02 16:18:27,319 - stpipe.step - INFO - Spec2Pipeline instance created.
2026-03-02 16:18:27,321 - stpipe.step - INFO - AssignWcsStep instance created.
2026-03-02 16:18:27,321 - stpipe.step - INFO - BadpixSelfcalStep instance created.
2026-03-02 16:18:27,322 - stpipe.step - INFO - MSAFlagOpenStep instance created.
2026-03-02 16:18:27,323 - stpipe.step - INFO - CleanFlickerNoiseStep instance created.
2026-03-02 16:18:27,324 - stpipe.step - INFO - BackgroundStep instance created.
2026-03-02 16:18:27,325 - stpipe.step - INFO - ImprintStep instance created.
2026-03-02 16:18:27,326 - stpipe.step - INFO - Extract2dStep instance created.
2026-03-02 16:18:27,329 - stpipe.step - INFO - MasterBackgroundMosStep instance created.
2026-03-02 16:18:27,330 - stpipe.step - INFO - FlatFieldStep instance created.
2026-03-02 16:18:27,330 - stpipe.step - INFO - PathLossStep instance created.
2026-03-02 16:18:27,331 - stpipe.step - INFO - BarShadowStep instance created.
2026-03-02 16:18:27,331 - stpi

RuntimeError: Traceback (most recent call last):
  File "/Users/morrison/github/stdatamodels/src/stdatamodels/properties.py", line 382, in __getattr__
    val = self._instance[attr]
          ~~~~~~~~~~~~~~^^^^^^
KeyError: 'photometry'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/morrison/github/jwst/jwst/pipeline/calwebb_spec2.py", line 156, in process
    result = self.process_exposure_product(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/morrison/github/jwst/jwst/pipeline/calwebb_spec2.py", line 368, in process_exposure_product
    calibrated = self._process_miri_wfss(calibrated)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/morrison/github/jwst/jwst/pipeline/calwebb_spec2.py", line 733, in _process_miri_wfss
    calibrated = self.photom.run(calibrated)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/morrison/github/jwst/jwst/stpipe/core.py", line 314, in run
 

In [ ]:
# extra tool on background reference file
background_file = 'MIRI_WFSS_bkg_February2026.fits'
backmodel = datamodels.open(background_file)
print(backmodel.meta.subarray.xsize, backmodel.meta.subarray.xstart)
print(backmodel.meta.subarray.ysize, backmodel.meta.subarray.ystart)

In [ ]:
# update the background reference file
update_background = False
if update_background:
    bkg_file = 'jwst_miri_bkg_0001.fits'
    model = datamodels.open(bkg_file)
    new_model = model.copy()
    new_model.meta.subarray.xsize = 1032
    new_model.meta.subarray.ysize = 1024
    new_model.meta.subarray.xstart = 1
    new_model.meta.subarray.ystart = 1
    new_model.save('jwst_miri_bkg_0001_new.fits')
    

In [ ]:
# Update the flat do that the lyot region is not used
update_flat = True
if update_flat: 
    flat_file = 'MIRI_WFSS_ref_flat_202602.fits'
    model = datamodels.open(flat_file)
    new_model = model.copy()
    dq = new_model.dq
    
    new_model.save('MIRI_WFSS_ref_flat_202602_v2_new.fits')
    

In [3]:
test_photom = True
if test_photom:
    photom_file = 'jwst_miri_photom_0231.fits'
    model = datamodels.MirWfssPhotomModel(photom_file)
    print(model)

<MirWfssPhotomModel from jwst_miri_photom_0231.fits>
